## מעבר ל-NumPy: to_numpy()

זו החוליה לשבוע 11: pandas טוב לקריאה, סינון וקיבוץ של נתונים גולמיים — אבל כל החישוב הכבד (הולכת שגיאות, התאמה ליניארית, שילוב עם broadcasting) חוזר להיות **מערכי NumPy**, כמו בשבועות 7–9. `().to_numpy` הוא הגשר.

In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv("launch_log.csv")

### דוגמה: חילוץ מדגם לניתוח נומרי

נניח שרוצים את כל ערכי הטווח שנמדדו בזווית 45°, כמערך NumPy רגיל — כדי לחשב עליו ישירות עם הכלים שכבר מכירים.

In [ ]:
range_45 = df[df["angle_deg"] == 45]["range_measured"].to_numpy()
print(type(range_45), range_45.shape)

# עכשיו אפשר להשתמש בכל כלי NumPy שכבר מכירים:
print("ממוצע (NumPy):", range_45.mean())
print("ממוצע (pandas ישירות, להשוואה):", df[df["angle_deg"] == 45]["range_measured"].mean())

### באג נפוץ: התאמת אינדקסים (Index Alignment) בין שני `Series`

זה ההבדל המושגי החשוב ביותר בין pandas ל-NumPy: כשמחסרים שני מערכי NumPy, ההתאמה היא **לפי מיקום**. כשמחסרים שני `Series` של pandas, ההתאמה היא **לפי index** — ואם שני ה-`Series` עברו סינון שונה (ויש להם index שונה), pandas "מיישר" אותם לפי תווית ה-index, לא לפי מיקום — והתוצאה מלאה ב-`NaN` איפה שאין התאמה.

In [ ]:
s1 = df[df["angle_deg"] == 45]["range_measured"]   # index: 18, 19, 20, 21, 22, 23 (לדוגמה)
s2 = df[df["angle_deg"] == 60]["range_measured"]   # index: 24, 25, 26, 27, 28, 29 - שונה!

diff_wrong = s1 - s2
print(diff_wrong.to_numpy())   # כולו NaN! אין אף index משותף בין שתי הקבוצות

`s1` ו-`s2` מגיעות מקבוצות **שונות לגמרי** של שורות, כך שאין אף תווית index משותפת ביניהן — pandas לא "מוצא" זוגות תואמים, ומחזיר `NaN` בכל מקום. הפתרון: להמיר קודם ל-NumPy עם `().to_numpy`, ואז ההתאמה חוזרת להיות לפי מיקום (ואחריות המתכנת לוודא שהאורכים תואמים ומשמעותיים).

In [ ]:
diff_right = s1.to_numpy() - s2.to_numpy()   # התאמה לפי מיקום, לא לפי index
print(diff_right)

### נסו בעצמכם

חלצו את `v0_measured` בזווית 30° כמערך NumPy, וחשבו את סטיית התקן שלו (`().std`) - השוו לתוצאה שמתקבלת ישירות מ-pandas (`()df[...]['v0_measured'].std`).

In [ ]:
# v0_30 = df[df["angle_deg"] == 30]["v0_measured"].to_numpy()
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
v0_30 = df[df["angle_deg"] == 30]["v0_measured"].to_numpy()
print("NumPy:", v0_30.std())
print("pandas:", df[df["angle_deg"] == 30]["v0_measured"].std())
```
```{note}
ייתכן הבדל קטן בין השתיים: `numpy.std` משתמש כברירת מחדל ב-`ddof=0` (חלוקה ב-n), בעוד ש-`pandas.Series.std` משתמש ב-`ddof=1` (חלוקה ב-n-1) — הבדל שנחזור אליו בשבוע 11.
```
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "מדוע חיסור בין שני מערכי NumPy מתנהג שונה מחיסור בין שני Series של pandas שעברו סינון שונה?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "NumPy מתאים לפי מיקום; pandas Series מתאים לפי index, שעלול להיות שונה בין שני ה-Series", "correct": True, "feedback": "נכון."},
            {"answer": "אין הבדל, זו אותה התנהגות בדיוק", "correct": False, "feedback": "לא — ראינו דוגמה שהתוצאה שונה לגמרי (NaN)."},
            {"answer": "pandas לא תומך בחיסור בכלל", "correct": False, "feedback": "לא נכון."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

חלצו את `range_measured` לזווית 75°, כמערך NumPy, וחשבו את הטווח המרבי פחות הטווח המזערי (`ptp`, "peak to peak") שנמדדו באותה זווית.

In [ ]:
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
range_75 = df[df["angle_deg"] == 75]["range_measured"].to_numpy()
print(range_75.max() - range_75.min())
```
`````